# Config and Dependencies

In [1]:
%pip install azure-cosmos requests tqdm sentence-transformers torch

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import os
import tarfile
import email
import re
from email.policy import default
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
from azure.cosmos import CosmosClient, PartitionKey, exceptions
import tqdm as notebook_tqdm

# =====================================================================
# CONFIGURATION
# =====================================================================
ENRON_TAR_URL = "https://www.cs.cmu.edu/~enron/enron_mail_20150507.tar.gz"
ENRON_TAR_FILENAME = "enron_mail_20150507.tar.gz"

# Azure Cosmos DB Config
COSMOS_ENDPOINT = os.getenv("COSMOS_ENDPOINT")
COSMOS_KEY = os.getenv("COSMOS_KEY")
COSMOS_DATABASE = os.getenv("COSMOS_DATABASE")
COSMOS_ENRON_COLLECTION = os.getenv("COSMOS_ENRON_COLLECTION")

# Safety boundaries: Keep smaller first to test API rate limits
MAX_EMAILS_TO_PROCESS = 10000 
BATCH_SIZE = 16  # Smaller batch size to prevent hitting cloud token-per-minute (TPM) limits

# Define Pipeline Helper Functions

In [4]:
def download_enron_dataset(url: str, filename: str) -> None:
    """Download the Enron dataset tarball.

    Args:
        url: URL for the `.tar.gz` archive.
        filename: Local file path where the archive should be saved.

    Returns:
        None
    """
    if os.path.exists(filename):
        print(f" Found cached dataset: {filename}. Skipping download.")
        return
    
    print(f" Downloading Enron dataset from {url}...")
    response = requests.get(url, stream=True)
    total_size = int(response.headers.get('content-length', 0))
    
    with open(filename, "wb") as f, tqdm(
        total=total_size, unit='B', unit_scale=True, desc="Downloading Tarball"
    ) as pbar:
        for data in response.iter_content(chunk_size=8192):
            f.write(data)
            pbar.update(len(data))


def extract_and_parse_emails(tar_path: str, max_items: int) -> list[dict]:
    """Parse raw MIME emails from a compressed tarball.

    The function streams the archive, extracts messages under maildir entries,
    normalizes the message body, and returns structured records ready for
    embedding generation.

    Args:
        tar_path: Local file system path to the Enron `.tar.gz` archive.
        max_items: Maximum number of email items to parse. Use None or 0
            to process the entire archive.

    Returns:
        A list of dictionaries containing email metadata and text payloads.
    """
    print(" Parsing raw emails directly from archive...")
    parsed_emails = []
    count = 0
    
    with tarfile.open(tar_path, "r:gz") as tar:
        for member in tqdm(tar, desc="Scanning Tarball Archive"):
            if member.isfile() and "maildir" in member.name:
                f = tar.extractfile(member)
                if f is None:
                    continue
                
                try:
                    raw_content = f.read().decode('utf-8', errors='ignore')
                    msg = email.message_from_string(raw_content, policy=default)
                    
                    if msg.is_multipart():
                        body = "".join(
                            part.get_payload(decode=True).decode('utf-8', errors='ignore')
                            for part in msg.walk() if part.get_content_type() == 'text/plain'
                        )
                    else:
                        body = msg.get_payload(decode=True).decode('utf-8', errors='ignore')

                    body = clean_email_text(body)
                    
                    subject = msg.get('Subject', '')
                    sender = msg.get('From', '')
                    to_recipients = msg.get('To', '')
                    date_sent = msg.get('Date', '')
                    
                    text_to_embed = f"Subject: {subject}\nFrom: {sender}\nBody: {body}"
                    
                    parsed_emails.append({
                        "id": f"enron_{count}",
                        "subject": subject,
                        "from": sender,
                        "to": to_recipients,
                        "date": date_sent,
                        "body": body[:3000],
                        "text_to_embed": text_to_embed
                    })
                    count += 1
                except Exception:
                    # Gracefully continue if a specific compressed file block is corrupted
                    continue
                
                if max_items and count >= max_items:
                    break
                    
    return parsed_emails


def clean_email_text(text):
    """Normalize email text for embedding and storage.

    Args:
        text: Raw email text to clean.

    Returns:
        The cleaned, whitespace-normalized, truncated text.
    """
    if not text:
        return ""
    # Replace non-breaking spaces with standard spaces
    text = text.replace('\xa0', ' ')
    # Convert tabs and hard returns to spaces, then collapse repeated whitespace
    text = re.sub(r'[\r\n\t]+', ' ', text)
    text = re.sub(r' {2,}', ' ', text)
    # Optional: Truncate to a safe length (e.g., ~8000 characters or use a proper token splitter)
    return text.strip()[:8000]

# Extract The Data

In [5]:
# Check if the tar file has already been downloaded
if not os.path.exists(ENRON_TAR_FILENAME):
    print(f"{ENRON_TAR_FILENAME} not found. Starting download...")
    download_enron_dataset(ENRON_TAR_URL, ENRON_TAR_FILENAME)
else:
    print(f"{ENRON_TAR_FILENAME} already exists. Skipping download.")

# Proceed with extraction and parsing
emails = extract_and_parse_emails(ENRON_TAR_FILENAME, MAX_EMAILS_TO_PROCESS)

# Apply cleaning to your dataset list
# Apply cleaning to each email dict: normalize the text_to_embed field and keep items as dicts
for e in emails:
    e['text_to_embed'] = clean_email_text(e.get('text_to_embed', e.get('body', '')))

# Use the list of cleaned email dicts for downstream processing
cleaned_emails = emails

print(f"\nReady to vectorize {len(cleaned_emails)} emails.")

enron_mail_20150507.tar.gz already exists. Skipping download.
 Parsing raw emails directly from archive...


Scanning Tarball Archive: 10169it [00:18, 555.12it/s]



Ready to vectorize 10000 emails.


# Generate Embeddings Locally

In [6]:
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

print("Initializing Local Embedding Model...")
# This will download the model once (~90MB) and cache it locally.
# It automatically handles truncation if strings are too long.
model = SentenceTransformer("all-MiniLM-L6-v2")

print("Computing embeddings locally...")
embeddings = []

# Process in batches to keep memory usage stable
for i in tqdm(range(0, len(cleaned_emails), BATCH_SIZE), desc="Local Embedding Generation"):
    batch_items = cleaned_emails[i:i + BATCH_SIZE]
    batch_texts = [item.get("text_to_embed", "") for item in batch_items]
    
    if not batch_texts:
        continue
      
    try:
        # Generate embeddings locally on your CPU (or GPU if available)
        batch_embeddings = model.encode(batch_texts, show_progress_bar=False)
        
        # Convert numpy arrays from the model into standard lists
        embeddings.extend([emb.tolist() for emb in batch_embeddings])
        
    except Exception as local_err:
        print(f"\n Error processing local embedding batch at range {i}-{i+BATCH_SIZE}: {local_err}")
        raise local_err

# Map local vectors back and clean up the dictionary payload
for item, emb in zip(cleaned_emails, embeddings):
    item["vector"] = emb
    if "text_to_embed" in item:
        del item["text_to_embed"]

print(" Local embedding synthesis complete!")

Initializing Local Embedding Model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3488.62it/s]


Computing embeddings locally...


Local Embedding Generation: 100%|██████████| 625/625 [05:33<00:00,  1.87it/s]

 Local embedding synthesis complete!


# Initialise CosmosDB and Ingest Data

In [9]:
print(" Establishing connection to Azure Cosmos DB...")
cosmos_client = CosmosClient(COSMOS_ENDPOINT, credential=COSMOS_KEY)
db = cosmos_client.create_database_if_not_exists(id=COSMOS_DATABASE)

# 'all-MiniLM-L6-v2' outputs vectors of length 384
EMBEDDING_DIMENSIONS = 384

vector_embedding_policy = {
    "vectorEmbeddings": [
        {
            "path": "/vector",
            "dataType": "float32",
            "distanceFunction": "cosine",
            "dimensions": EMBEDDING_DIMENSIONS
        }
    ]
}

indexing_policy = {
    "includedPaths": [{"path": "/*"}],
    "excludedPaths": [{"path": "/vector/*"}],
    "vectorIndexes": [
        {
            "path": "/vector",
            "type": "quantizedFlat"
        }
    ]
}

try:
    container = db.create_container_if_not_exists(
        id=COSMOS_ENRON_COLLECTION,
        partition_key=PartitionKey(path="/id"),
        indexing_policy=indexing_policy,
        vector_embedding_policy=vector_embedding_policy
    )
    print(f" Target vector-enabled container ready: {COSMOS_ENRON_COLLECTION}")
except exceptions.CosmosHttpResponseError as err:
    print(f" Container generation failed: {err}")
    raise err

# Bulk Delete Existing Items in the Container
print(f" Deleting existing items in container {COSMOS_ENRON_COLLECTION}...")
try:
    existing_items = list(container.read_all_items())
    for item in tqdm(existing_items, desc="Deleting Existing Items"):
        item_id = item["id"]
        try:
            container.delete_item(item=item_id, partition_key=item_id)
        except exceptions.CosmosHttpResponseError as delete_err:
            if getattr(delete_err, "status_code", None) == 404:
                continue
            raise delete_err
    print(f" Successfully deleted {len(existing_items)} existing items.")
except exceptions.CosmosHttpResponseError as delete_err:
    print(f" Error during deletion of existing items: {delete_err}")
    raise delete_err

# Bulk upload loop with inline status monitoring
print(f" Commencing container ingestion...")
for doc in tqdm(emails, desc="Cosmos DB Upsert Status"):
    try:
        container.upsert_item(body=doc)
    except exceptions.CosmosHttpResponseError as upload_err:
        print(f" Failed to upload document {doc['id']}: {upload_err}")

print("\n Ingestion loop finished successfully.")

 Establishing connection to Azure Cosmos DB...
 Target vector-enabled container ready: EnronEmailVectorStore
 Deleting existing items in container EnronEmailVectorStore...


Deleting Existing Items: 100%|██████████| 5624/5624 [33:46<00:00,  2.77it/s]


 Successfully deleted 5624 existing items.
 Commencing container ingestion...


Cosmos DB Upsert Status: 100%|██████████| 10000/10000 [1:06:57<00:00,  2.49it/s]


 Ingestion loop finished successfully.
